In [ ]:
from scripts.ingest_data
df = load_data()

ModuleNotFoundError: No module named 'scripts'

In [31]:
from ingest_data import load_data

docs = load_data()

In [32]:
for idx, item in enumerate(docs):
    item['id'] = idx

In [33]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [34]:
data_gen_instructions = """
You are generating search queries to test a coffee search engine across different difficulty levels.

Given the following coffee review record, generate exactly 5 distinct search queries or questions that a user might type to find this coffee.

Difficulty Breakdown:
1. Query 1 (Easy - Direct Metadata):
   - Combine exact metadata fields (e.g., roaster name, country, rating, origin).
   - Tests basic keyword filtering and metadata retrieval.

2. Queries 2 & 3 (Medium - Semantic & Paraphrased):
   - Describe flavor notes, mouthfeel, or brewing behavior using SYNONYMS and PARAPHRASING.
   - Do NOT use exact distinctive words from the text (e.g., replace "narcissus" with "floral notes", replace "syrupy mouthfeel" with "heavy body").
   - Tests vector embedding quality and semantic matching.

3. Queries 4 & 5 (Hard - Implicit / Broad User Need):
   - Write a short, realistic query representing a user with a preference, without naming specific regions or exact roasters.
   - Example: "Complex espresso blend that cuts through milk with chocolate and fruit notes" or "High-scoring light roast with a heavy body"
   - Tests ranking performance when information is partial or incomplete.

Formatting: Return exactly 5 clear, complete, self-contained search queries labeled 1 through 5.
""".strip()

In [37]:
from openai import OpenAI
from evaluation_utils import llm_structured_retry
import json

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        OpenAI(),
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [38]:
docs = docs[:30]

In [40]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, docs, generate_ground_truth)

  0%|          | 0/30 [00:00<?, ?it/s]

In [41]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

154

In [57]:
for doc in ground_truth:
    if doc["question"].strip() == "":
        ground_truth.remove(doc)

In [62]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.03105

In [64]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [65]:
df_ground_truth.to_csv("dataset/ground_truth-new.csv", index=False)

In [9]:
from ingest_data import load_data, build_index

data = load_data()
for idx, item in enumerate(data):
        item['id'] = idx
index = build_index(data)

In [67]:
def text_search(query):
    boost_dict = {'desc_1': 2, 'desc_2': 1.5, 'desc_3': 1.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [69]:
from assistant import create_assistant


Response(id='resp_0fbbba579e3682b1006a5f703089948198ac72aac8a15171ac', created_at=1784639536.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0fbbba579e3682b1006a5f7030f94c8198b6fce67ab29a7758', content=[ResponseOutputText(annotations=[], text='**A.R.C. “Sweety” Espresso Blend**\n- **Origin:** Panama + Ethiopia\n- **Roast Level:** Medium-Light\n- **Quality Rating:** 95\n- **Style:** Espresso blend\n\n**What it tastes like:**\n- **Core notes:** Sweet-toned, deeply rich, chocolaty\n- **Aroma/cup:** Vanilla paste, dark chocolate, narcissus, pink grapefruit zest, black cherry\n- **Mouthfeel:** Plush, syrupy\n- **Finish:** Resonant, flavor-saturated\n- **In milk:** Chocolate deepens, with vanilla paste, black cherry, and floral citrus zest\n\n**Extraction advice:**\n- Best as a **radiant espresso** that works well **straight or in milk**\n- Expect **rich dark chocolate** and **bl

In [99]:
q = ground_truth[15]
q

{'question': '1. Roast House Ethiopia Suke Quto Medium-Light from United States, origin Guji Zone Oromia Region, rating 92',
 'document': 3}

In [100]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [122]:
from tqdm.auto import tqdm

def calc_relevance(q, search_function=text_search):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def calc_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = calc_relevance(q, search_function)
        relevance_total.append(relevance)
    
    return relevance_total

In [124]:
relevance_total_text = calc_relevance_total(ground_truth, text_search)

  0%|          | 0/152 [00:00<?, ?it/s]

In [125]:
def hit_rate(relevance):
    count = 0

    for line in relevance:
        if 1 in line:
            count += 1
    
    return count/len(relevance)

hit_rate(relevance_total_text)

0.34868421052631576

In [126]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1/(rank + 1)
                break
    
    return total_score/len(relevance)

mrr(relevance_total_text)

0.2594298245614035

In [127]:
def evaluate(ground_truth, search_function):
    relevance_total = calc_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [6]:
import pandas as pd
df_ground_truth = pd.read_csv("dataset/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [10]:
from evaluate import evaluate_results

def text_search(query):
    boost_dict = {'desc_1': 2, 'desc_2': 1.5, 'desc_3': 1.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

evaluate_results(ground_truth, text_search)

  0%|          | 0/152 [00:00<?, ?it/s]

{'hit_rate': 0.34868421052631576, 'mrr': 0.2594298245614035}